In [ ]:
# --- Notebook environment bootstrap ---
# Purpose:
# 1) Resolve project root robustly
# 2) Standardize working directory + import paths
# 3) Enable autoreload in Jupyter
# 4) Import all dependencies used across notebook cells

# Standard library imports
import os
import sys
from pathlib import Path
import datetime as dt
from functools import partial
import itertools
import math
import multiprocessing
import pickle
import random
import warnings
import concurrent.futures

# Resolve project root and normalize paths
def resolve_project_root(markers=(".git", "Program")):
    """
    Walk upward from the current directory and return the first folder
    containing any marker (e.g., '.git' or 'Program').
    """
    cwd = Path.cwd()
    return next(
        (p for p in (cwd, *cwd.parents) if any((p / m).exists() for m in markers)),
        cwd,
    )

def configure_paths(project_root: Path):
    """
    Set working directory to project root and prepend import paths so
    local modules are preferred over site-packages with same names.
    """
    os.chdir(project_root)

    root_str = str(project_root)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)

    program = project_root / "Program"
    program_str = str(program)
    if program_str not in sys.path:
        sys.path.insert(0, program_str)

    return program

# Respect existing notebook variables if they already exist
if "markers" not in globals():
    markers = (".git", "Program")
if "current_dir" not in globals():
    current_dir = Path.cwd()
if "project_root" not in globals() or not isinstance(project_root, Path):
    project_root = resolve_project_root(markers=markers)

program_path = configure_paths(project_root)

# Jupyter autoreload (safe no-op outside IPython)
try:
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("load_ext", "autoreload")
        ip.run_line_magic("autoreload", "2")
except Exception:
    pass

# Third-party imports
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.stats import (
    false_discovery_control,
    kendalltau,
    linregress,
    pearsonr,
    spearmanr,
    ttest_ind,
    wilcoxon,
)
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, pacf, adfuller, coint, grangercausalitytests
from arch import arch_model
from tqdm import tqdm

# Project imports
from fundamentals import *
from helper_functions import *
from plot import *
from stock_screener import *
from technicals import *

# External data clients
try:
    from tvDatafeed import TvDatafeed, Interval
    tv = TvDatafeed()
except Exception:
    tv = None

warnings.filterwarnings("ignore")

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = True
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HSI", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the current date
current_date = modify_current_date(start, index_name)

In [ ]:
# Read the Shiller data
filename = "Program/shiller_data.xls"
shiller_data = pd.read_excel(filename, sheet_name="Data", skiprows=7)

In [ ]:
def clean_data(data, cols_keep, drop_rows=-2):
    """"
    Cleans the Shiller data by dropping unnecessary rows and columns.

    Parameters:
    - data (pd.DataFrame): The raw Shiller data.
    - cols_keep (dict): A dictionary mapping old column names to new ones.
    - drop_rows (int, optional): Number of rows to drop from the end of the DataFrame. Default is -2.

    Returns:
    - data (pd.DataFrame): The cleaned DataFrame with renamed columns and formatted date.
    """

    # Drop the last rows as they may contain partial or incomplete data
    data = data.iloc[:drop_rows]

    # Check if there are any unnamed columns
    unnamed_cols = [col for col in data.columns if "Unnamed" in str(col)]
    if unnamed_cols:
        # Drop the unnamed columns
        data = data.drop(columns=unnamed_cols)
    
    # Filter and rename the columns
    data = data[list(cols_keep.keys())].rename(columns=cols_keep)

    # Convert the date format from decimal year to datetime
    data["Date"] = pd.to_datetime(data["Date"].apply(lambda x: f"{int(x)}-{int((x % 1) * 100 + 1):02d}-01" if pd.notnull(x) else None))

    # Set the Date column as index
    data.set_index("Date", inplace=True)

    return data

# Select only the specified columns with descriptive names
cols_keep = {
    "Date": "Date",
    "P": "Price",
    "E": "Earnings",
    "Rate GS10": "Long Interest Rate GS10",
    "Price": "Real Price",
    "Earnings": "Real Earnings",
    "Earnings.1": "Real TR Scaled Earnings",
    "CAPE": "CAPE",
    "TR CAPE": "TR CAPE",
    "Yield": "Excess CAPE Yield",
    "Real Return": "10 Year Annualized Stock Real Return",
    "Real Return.1": "10 Year Annualized Bonds Real Return"
}

# Clean the data
shiller_data = clean_data(shiller_data, cols_keep)

In [ ]:
# Calculate the mean and standard deviation of CAPE and Excess CAPE Yield
shiller_data["CAPE Mean"] = shiller_data["CAPE"].expanding().mean()
shiller_data["CAPE SD"] = shiller_data["CAPE"].expanding().std()
shiller_data["CAPE Z-Score"] = (shiller_data["CAPE"] - shiller_data["CAPE Mean"]) / shiller_data["CAPE SD"]
shiller_data["Excess CAPE Yield Mean"] = shiller_data["Excess CAPE Yield"].expanding().mean()
shiller_data["Excess CAPE Yield SD"] = shiller_data["Excess CAPE Yield"].expanding().std()
shiller_data["Excess CAPE Yield Z-Score"] = (shiller_data["Excess CAPE Yield"] - shiller_data["Excess CAPE Yield Mean"]) / shiller_data["Excess CAPE Yield SD"]

In [ ]:
# Filter out rows with NaN values in CAPE and 10 Year Annualized Stock Real Return
cape_real_return_10y = shiller_data.dropna(subset=["CAPE", "10 Year Annualized Stock Real Return"])

# Create a scatter plot for 10-Year Annualized Stock Real Return vs CAPE
plt.figure(figsize=(12, 8))
plt.scatter(cape_real_return_10y["CAPE"], cape_real_return_10y["10 Year Annualized Stock Real Return"], alpha=0.6)

# Add trendline
z = np.polyfit(cape_real_return_10y["CAPE"], cape_real_return_10y["10 Year Annualized Stock Real Return"], 1)
p = np.poly1d(z)
plt.plot(cape_real_return_10y["CAPE"], p(cape_real_return_10y["CAPE"]), color="red", linestyle="--", linewidth=2)

# Set the labels and title
plt.xlabel("CAPE", fontsize=12)
plt.ylabel("10-Year Annualized Stock Real Return", fontsize=12)
plt.title("10-Year Annualized Stock Real Return vs CAPE", fontsize=14)

# Add correlation coefficient
correlation = cape_real_return_10y["CAPE"].corr(cape_real_return_10y["10 Year Annualized Stock Real Return"])
plt.annotate(f"Correlation: {correlation:.3f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12)

# Add vertical line for current CAPE
current_cape = shiller_data["CAPE"].iloc[-1]
plt.axvline(x=current_cape, color="blue", linestyle="-")
plt.text(current_cape + 1, plt.ylim()[0] + 0.02, f"Current CAPE: {current_cape:.2f}", rotation=90, verticalalignment="bottom")

# Calculate the mean CAPE
mean_cape = cape_real_return_10y["CAPE"].mean()

# Add region labels
plt.text(cape_real_return_10y["CAPE"].max()*0.75, cape_real_return_10y["10 Year Annualized Stock Real Return"].max()*0.8, 
         "Overvalued\nGood Returns", ha="center", fontsize=12, bbox=dict(facecolor="yellow", alpha=0.5))
plt.text(cape_real_return_10y["CAPE"].min()*1.5, cape_real_return_10y["10 Year Annualized Stock Real Return"].max()*0.8, 
         "Undervalued\nGood Returns", ha="center", fontsize=12, bbox=dict(facecolor="lightgreen", alpha=0.5))
plt.text(cape_real_return_10y["CAPE"].max()*0.75, cape_real_return_10y["10 Year Annualized Stock Real Return"].min()*0.5, 
         "Overvalued\nPoor Returns", ha="center", fontsize=12, bbox=dict(facecolor="lightsalmon", alpha=0.5))
plt.text(cape_real_return_10y["CAPE"].min()*1.5, cape_real_return_10y["10 Year Annualized Stock Real Return"].min()*0.5, 
         "Undervalued\nPoor Returns", ha="center", fontsize=12, bbox=dict(facecolor="yellow", alpha=0.5))

# Adjust the spacing
plt.tight_layout()

# Show the plot
plt.show()

# Calculate the expected 10-Year Annualized Real Return based on the current CAPE
expected_real_return_10y = p(current_cape)
print(f"Current CAPE: {current_cape:.4f}")
print(f"Expected 10-Year Annualized Real Return based on historical relationship: {expected_real_return_10y:.4f}")
print(f"Historical average CAPE: {mean_cape:.4f}")

In [ ]:
# Filter out rows with NaN values in Excess CAPE Yield and 10 Year Annualized Stock Real Return
excess_yield_real_return_10y = shiller_data.dropna(subset=["Excess CAPE Yield", "10 Year Annualized Stock Real Return"])

# Create a scatter plot for 10-Year Annualized Stock Real Return vs Excess CAPE Yield
plt.figure(figsize=(12, 8))
plt.scatter(excess_yield_real_return_10y["Excess CAPE Yield"], excess_yield_real_return_10y["10 Year Annualized Stock Real Return"], alpha=0.6)

# Add trendline
z = np.polyfit(excess_yield_real_return_10y["Excess CAPE Yield"], excess_yield_real_return_10y["10 Year Annualized Stock Real Return"], 1)
p = np.poly1d(z)
plt.plot(excess_yield_real_return_10y["Excess CAPE Yield"], p(excess_yield_real_return_10y["Excess CAPE Yield"]), color="red", linestyle="--", linewidth=2)

# Set the labels and title
plt.xlabel("Excess CAPE Yield", fontsize=12)
plt.ylabel("10 Year Annualized Stock Real Return", fontsize=12)
plt.title("10-Year Annualized Stock Real Return vs Excess CAPE Yield", fontsize=14)

# Add correlation coefficient
correlation = excess_yield_real_return_10y["Excess CAPE Yield"].corr(excess_yield_real_return_10y["10 Year Annualized Stock Real Return"])
plt.annotate(f"Correlation: {correlation:.3f}", xy=(0.95, 0.95), xycoords="axes fraction", fontsize=12, ha="right")

# Add vertical line for current Excess CAPE Yield
current_excess_yield = shiller_data["Excess CAPE Yield"].iloc[-1]
plt.axvline(x=current_excess_yield, color="blue", linestyle="-")
plt.text(current_excess_yield + 0.01, plt.ylim()[0] + 0.02, f"Current Excess CAPE Yield: {current_excess_yield:.2f}", rotation=90, verticalalignment="bottom")

# Calculate the mean Excess CAPE Yield
mean_excess_yield = excess_yield_real_return_10y["Excess CAPE Yield"].mean()

# Add region labels
plt.text(excess_yield_real_return_10y["Excess CAPE Yield"].max()*0.8, excess_yield_real_return_10y["10 Year Annualized Stock Real Return"].max()*0.8, 
         "Undervalued\nGood Returns", ha="center", fontsize=12, bbox=dict(facecolor="lightgreen", alpha=0.5))
plt.text(excess_yield_real_return_10y["Excess CAPE Yield"].min()*0.6, excess_yield_real_return_10y["10 Year Annualized Stock Real Return"].max()*0.8,
            "Overvalued\nGood Returns", ha="center", fontsize=12, bbox=dict(facecolor="yellow", alpha=0.5))
plt.text(excess_yield_real_return_10y["Excess CAPE Yield"].max()*0.8, excess_yield_real_return_10y["10 Year Annualized Stock Real Return"].min()*0.5,
            "Undervalued\nPoor Returns", ha="center", fontsize=12, bbox=dict(facecolor="yellow", alpha=0.5))
plt.text(excess_yield_real_return_10y["Excess CAPE Yield"].min()*0.6, excess_yield_real_return_10y["10 Year Annualized Stock Real Return"].min()*0.5,
            "Overvalued\nPoor Returns", ha="center", fontsize=12, bbox=dict(facecolor="lightsalmon", alpha=0.5))

# Adjust the spacing
plt.tight_layout()

# Show the plot
plt.show()

# Calculate the expected 10-Year Annualized Real Return based on the current Excess CAPE Yield
expected_real_return_10y = p(current_excess_yield)
print(f"Current Excess CAPE Yield: {current_excess_yield:.4f}")
print(f"Expected 10-Year Annualized Real Return based on historical relationship: {expected_real_return_10y:.4f}")
print(f"Historical average Excess CAPE Yield: {mean_excess_yield:.4f}")

In [ ]:
# Get the price data of the index
sp500_df = get_df("^GSPC", current_date, max_period=True)

# Find bear markets in the S&P 500 data
bear_df, bull_df, bear_starts, bear_ends, bull_starts, bull_ends = find_market_cycles(sp500_df)

# Create a figure
plt.figure(figsize=(10, 6))

# Plot the price
plt.plot(sp500_df.index, sp500_df["Close"], label="S&P 500 Close")

# Shade bear market periods
for start, end in zip(bear_starts, bear_ends):
    plt.axvspan(start, end, alpha=0.3, color="red", label="Bear market" if start == bear_starts[0] else "")

# Set the x limits
plt.xlim(sp500_df.index.min(), sp500_df.index.max())

# Set the labels and title
plt.title("Closing price history of S&P 500 with Bear Markets", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Price", fontsize=12)

# Use logarithmic scale for better visualization of historical data
plt.yscale("log")

# Add a legend
plt.legend()

# Adjust the spacing
plt.tight_layout()

# Show the plot
plt.show()

# Display the DataFrame
display(bear_df)

In [ ]:
# Calculate future 10-year and 1-year annualized returns based on historical data

# Common calculation function to avoid code duplication
def calculate_forward_returns(df, years, trading_days_per_year=252):
    """"
    Function to calculate future returns for a given number of years.

    Parameters:
    - df (pd.DataFrame): DataFrame containing S&P 500 data with a DateTime index.
    - years (int): Number of years to calculate future returns for.
    - trading_days_per_year (int, optional): Number of trading days in a year. Default is 252.

    Returns:
    - forward_returns (pd.DataFrame): DataFrame containing future returns for the specified number of years.
    """

    # Ensure the DataFrame is sorted by date
    days = int(years * trading_days_per_year)
    dates_to_use = df.index[:-days]
    target_dates = dates_to_use + pd.DateOffset(years=years)
    
    # Find the closest available future date in our dataset
    indices = df.index.searchsorted(target_dates)
    
    # Filter out dates where the target is beyond our dataset
    valid = indices < len(df.index)
    dates_to_use = dates_to_use[valid]
    indices = indices[valid]
    
    # Get the prices for calculation
    future_dates = df.index[indices]
    start_prices = df.loc[dates_to_use, "Close"].values
    end_prices = df.iloc[indices]["Close"].values
    
    # Calculate actual time elapsed in years for accurate annualization
    years_elapsed = (future_dates - dates_to_use).days / 365.25
    
    # Calculate annualized returns
    annualized_return = (end_prices / start_prices) ** (1 / years_elapsed) - 1

    # Create a DataFrame to hold the results
    forward_returns = pd.DataFrame({f"Future {years}Y Return": annualized_return * 100}, index=dates_to_use)
    
    return forward_returns

# Calculate both return periods
future_returns_10y_df = calculate_forward_returns(sp500_df, 10)
future_returns_1y_df = calculate_forward_returns(sp500_df, 1)

# Convert both to monthly frequency to match Shiller data
forward_returns_monthly_10y = future_returns_10y_df.resample("ME").last()
forward_returns_monthly_1y = future_returns_1y_df.resample("ME").last()

# Merge the returns first
combined_returns = pd.merge(
    forward_returns_monthly_10y,
    forward_returns_monthly_1y,
    left_index=True,
    right_index=True,
    how="outer"
)

# Then merge with Shiller data
analysis_df = pd.merge_asof(
    combined_returns.reset_index(),
    shiller_data[["Excess CAPE Yield Z-Score"]].reset_index(),
    left_on="Date",
    right_on="Date", 
    direction="backward"
).set_index("Date")

# Remove any rows with missing values
analysis_df = analysis_df.dropna()

# Create figure with two subplots sharing x-axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=False)

# 10-YEAR RETURNS PLOT
ax1.scatter(analysis_df["Excess CAPE Yield Z-Score"], analysis_df["Future 10Y Return"], alpha=0.6)

# Add a trendline
z_10y = np.polyfit(analysis_df["Excess CAPE Yield Z-Score"], analysis_df["Future 10Y Return"], 1)
p_10y = np.poly1d(z_10y)
ax1.plot(analysis_df["Excess CAPE Yield Z-Score"], p_10y(analysis_df["Excess CAPE Yield Z-Score"]), 
         color="red", linestyle="--", linewidth=2)

# Add correlation coefficient
corr_10y = analysis_df["Excess CAPE Yield Z-Score"].corr(analysis_df["Future 10Y Return"])
ax1.annotate(f"Correlation: {corr_10y:.3f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12)

# Add current Excess CAPE Yield Z-Score as vertical line
if not shiller_data["Excess CAPE Yield Z-Score"].isna().all():
    current_z_score = shiller_data["Excess CAPE Yield Z-Score"].iloc[-1]
    ax1.axvline(x=current_z_score, color="blue", linestyle="-", linewidth=2)
    ax1.text(current_z_score + 0.1, ax1.get_ylim()[0] + 2, f"Current: {current_z_score:.2f}", 
             rotation=90, color="blue")

# Set labels
ax1.set_xlabel("Excess CAPE Yield Z-Score", fontsize=12)
ax1.set_ylabel("Future 10-Year Annualized Return (%)", fontsize=12)
ax1.set_title("Future 10-Year Stock Return vs Excess CAPE Yield Z-Score", fontsize=14)

# Divide the plot into quadrants with labels
x_min, x_max = ax1.get_xlim()
y_min, y_max = ax1.get_ylim()
ax1.text(x_max*0.8, y_max*0.8, "Undervalued\nGood Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="lightgreen", alpha=0.5))
ax1.text(x_min*0.8, y_max*0.8, "Overvalued\nGood Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="yellow", alpha=0.5))
ax1.text(x_max*0.8, y_min*0.8, "Undervalued\nPoor Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="yellow", alpha=0.5))
ax1.text(x_min*0.8, y_min*0.8, "Overvalued\nPoor Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="lightsalmon", alpha=0.5))

# 1-YEAR RETURNS PLOT
ax2.scatter(analysis_df["Excess CAPE Yield Z-Score"], analysis_df["Future 1Y Return"], alpha=0.6)

# Add a trendline
z_1y = np.polyfit(analysis_df["Excess CAPE Yield Z-Score"], analysis_df["Future 1Y Return"], 1)
p_1y = np.poly1d(z_1y)
ax2.plot(analysis_df["Excess CAPE Yield Z-Score"], p_1y(analysis_df["Excess CAPE Yield Z-Score"]), 
         color="red", linestyle="--", linewidth=2)

# Add correlation coefficient
corr_1y = analysis_df["Excess CAPE Yield Z-Score"].corr(analysis_df["Future 1Y Return"])
ax2.annotate(f"Correlation: {corr_1y:.3f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12)

# Add current Excess CAPE Yield Z-Score as vertical line
if not shiller_data["Excess CAPE Yield Z-Score"].isna().all():
    ax2.axvline(x=current_z_score, color="blue", linestyle="-", linewidth=2)
    ax2.text(current_z_score + 0.1, ax2.get_ylim()[0] + 2, f"Current: {current_z_score:.2f}", 
             rotation=90, color="blue")

# Set labels
ax2.set_xlabel("Excess CAPE Yield Z-Score", fontsize=12)
ax2.set_ylabel("Future 1-Year Return (%)", fontsize=12)
ax2.set_title("Future 1-Year Stock Return vs Excess CAPE Yield Z-Score", fontsize=14)

# Divide the plot into quadrants with labels
x_min, x_max = ax2.get_xlim()
y_min, y_max = ax2.get_ylim()
ax2.text(x_max*0.8, y_max*0.8, "Undervalued\nGood Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="lightgreen", alpha=0.5))
ax2.text(x_min*0.8, y_max*0.8, "Overvalued\nGood Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="yellow", alpha=0.5))
ax2.text(x_max*0.8, y_min*0.8, "Undervalued\nPoor Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="yellow", alpha=0.5))
ax2.text(x_min*0.8, y_min*0.8, "Overvalued\nPoor Returns", ha="center", fontsize=10, 
         bbox=dict(facecolor="lightsalmon", alpha=0.5))

# Adjust layout
plt.tight_layout()

# Show the plots
plt.show()

# Display the expected returns based on the current excess cape yield z-score
current_z_score = shiller_data["Excess CAPE Yield Z-Score"].iloc[-1]
expected_10y_return = p_10y(current_z_score)
expected_1y_return = p_1y(current_z_score)
print(f"Current Excess CAPE Yield Z-Score: {current_z_score:.2f}")
print(f"Expected 10-Year Annualized Return: {expected_10y_return:.2f}%")
print(f"Expected 1-Year Return: {expected_1y_return:.2f}%")

In [ ]:
# Create a new DataFrame for cumulative correlation
cumulative_corr_df = pd.DataFrame(index=analysis_df.index)

# Calculate correlation coefficient using increasingly larger portions of the dataset
for i in range(60, len(analysis_df) + 1, 1):
    subset = analysis_df.iloc[:i]
    correlation = subset["Excess CAPE Yield Z-Score"].corr(subset["Future 10Y Return"])
    cumulative_corr_df.loc[subset.index[-1], "Cumulative Correlation"] = correlation
    
# Forward fill to have values for all dates
cumulative_corr_df = cumulative_corr_df.ffill()

# Plot the cumulative correlation
plt.figure(figsize=(10, 6))
plt.plot(cumulative_corr_df.index, cumulative_corr_df["Cumulative Correlation"], label="Cumulative Correlation")
plt.axhline(y=correlation, color="red", linestyle="-", alpha=0.7, label=f"Overall Correlation: {correlation:.3f}")

# Set the title and labels
plt.title("Cumulative Correlation: Excess CAPE Yield Z-Score vs Future 10Y Returns", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Correlation Coefficient", fontsize=12)

# Add legend
plt.legend()

# Adjust the spacing
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Create a time series plot with two y-axes
fig, ax1 = plt.subplots(figsize=(12, 7))

# Resample the Shiller data to daily frequency with forward-fill for better alignment with bear market dates
shiller_data_daily = shiller_data.resample("D").ffill()

# Plot the 10-year annualized stock real return
ax1.set_xlabel("Year", fontsize=12)
ax1.set_ylabel("10-Year Annualized Stock Real Return", color="tab:blue", fontsize=12)
ax1.plot(shiller_data_daily.index, shiller_data_daily["10 Year Annualized Stock Real Return"], color="tab:blue", label="10-Year Annualized Stock Real Return")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Create a second y-axis for Excess CAPE Yield
ax2 = ax1.twinx()
ax2.set_ylabel("Excess CAPE Yield", color="tab:red", fontsize=12)
ax2.plot(shiller_data_daily.index, shiller_data_daily["Excess CAPE Yield"], color="tab:red", label="Excess CAPE Yield")

# Plot the Excess CAPE Yield Mean
ax2.plot(shiller_data_daily.index, shiller_data_daily["Excess CAPE Yield Mean"], color="purple", linestyle="-", label="Excess CAPE Yield Mean")

# Plot +/- 1 sigma
ax2.plot(shiller_data_daily.index, shiller_data_daily["Excess CAPE Yield Mean"] + shiller_data_daily["Excess CAPE Yield SD"], color="red", linestyle="--", label=r"$\pm 1\sigma$")
ax2.plot(shiller_data_daily.index, shiller_data_daily["Excess CAPE Yield Mean"] - shiller_data_daily["Excess CAPE Yield SD"], color="red", linestyle="--")

# Plot +/- 2 sigma
ax2.plot(shiller_data_daily.index, shiller_data_daily["Excess CAPE Yield Mean"] + 2 * shiller_data_daily["Excess CAPE Yield SD"], color="red", linestyle=":", label=r"$\pm 2\sigma$")
ax2.plot(shiller_data_daily.index, shiller_data_daily["Excess CAPE Yield Mean"] - 2 * shiller_data_daily["Excess CAPE Yield SD"], color="red", linestyle=":")
ax2.tick_params(axis="y", labelcolor="tab:red")

# Get y-axis limits for both axes
min_y1, max_y1 = ax1.get_ylim()
min_y2, max_y2 = ax2.get_ylim()

# Mark bear market start dates
for i, start_date in enumerate(bear_starts):
    if start_date >= shiller_data_daily.index.min() and start_date <= shiller_data_daily.index.max():
        ax1.axvline(x=start_date, color="darkred", linestyle="-", alpha=0.5, linewidth=1)
        if i == 0:
            ax1.axvline(x=start_date, color="darkred", linestyle="-", alpha=0.5, linewidth=1, label="Bear Market Start")

# Add title
plt.title("10-Year Annualized Stock Real Return and Excess CAPE Yield Over Time", fontsize=14)

# Create combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best", fontsize=10)

# Add text annotation about bear markets
plt.figtext(0.01, 0.01, f"Number of Bear Markets shown: {len(bear_starts)}", fontsize=9)

# Set x-axis limits to focus on the data period
ax1.set_xlim(shiller_data_daily.index.min(), shiller_data_daily.index.max())

# Adjust the spacing
fig.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Create a time series plot with two y-axes for future returns and Excess CAPE Yield Z-Score
fig, ax1 = plt.subplots(figsize=(12, 7))

# Resample to daily frequency with forward-fill
analysis_df_daily = analysis_df.resample("D").ffill()

# Calculate correlations
corr_10y = analysis_df_daily["Excess CAPE Yield Z-Score"].corr(analysis_df_daily["Future 10Y Return"])
corr_1y = analysis_df_daily["Excess CAPE Yield Z-Score"].corr(analysis_df_daily["Future 1Y Return"])

# Plot Future Returns on left y-axis
ax1.set_xlabel("Year", fontsize=12)
ax1.set_ylabel("Future Returns (%)", color="tab:blue", fontsize=12)
ax1.plot(analysis_df_daily.index, analysis_df_daily["Future 10Y Return"], 
         color="tab:blue", label="Future 10-Year Return", linewidth=1.5)
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Second y-axis for Excess CAPE Yield Z-Score
ax2 = ax1.twinx()
ax2.set_ylabel("Excess CAPE Yield Z-Score", color="tab:red", fontsize=12)
ax2.plot(analysis_df_daily.index, analysis_df_daily["Excess CAPE Yield Z-Score"], 
         color="tab:red", label="Excess CAPE Yield Z-Score", linewidth=1.5)
ax2.tick_params(axis="y", labelcolor="tab:red")

# Mark bear market start dates
for i, start_date in enumerate(bear_starts):
    if analysis_df_daily.index.min() <= start_date <= analysis_df_daily.index.max():
        label = "Bear Market Start" if i == 0 else None
        ax1.axvline(x=start_date, color="darkred", linestyle="-", alpha=0.5, linewidth=1, label=label)

# Set the title
plt.title("Future Stock Returns vs Excess CAPE Yield Z-Score", fontsize=14, pad=15)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best", fontsize=10, framealpha=0.8)

# Add correlation annotation for both 10Y and 1Y
plt.figtext(0.01, 0.01, f"10-Year Correlation: {corr_10y:.3f}, 1-Year Correlation: {corr_1y:.3f}, Number of Bear Markets shown: {len(bear_starts)}",fontsize=10)

# Set x-axis limits
ax1.set_xlim(analysis_df_daily.index.min(), analysis_df_daily.index.max())

# Adjust layout
fig.tight_layout()
fig.subplots_adjust(bottom=0.1)

# Show the plot
plt.show()

In [ ]:
def adf_test(series, series_name, significance_level=0.05):
    """
    Perform Augmented Dickey-Fuller test to check stationarity.
    
    Parameters:
    - series (pandas Series or numpy array): Time series data to test.
    - series_name (str): Name of the series for display.
    - significance_level (float, optional): Significance level for the test. Default is 0.05.
    
    Returns:
    bool: True if series is stationary, False otherwise
    """

    # Drop NaN values
    series = series[~np.isnan(series)]
    
    # Perform Augmented Dickey-Fuller test
    result = adfuller(series)
    
    # Extract results
    adf_stat = result[0]
    p_value = result[1]
    critical_values = result[4]
    
    # Print detailed results
    print(f"ADF Test for {series_name}")
    print(f"ADF Statistic: {adf_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print("Critical Values:")
    for key, value in critical_values.items():
        print(f"{key}: {value:.4f}")
    
    # Determine stationarity
    is_stationary = p_value < significance_level
    conclusion = "Stationary (reject unit root)" if is_stationary else "Non-stationary (failed to reject unit root)"
    print(f"Conclusion at {significance_level} significance level: {conclusion}\n")
    
    return is_stationary

def difference_series(series, order=1):
    """
    Apply differencing to a time series.
    
    Parameters:
    - series (array-like): The time series to difference.
    - order (int, optional): The order of differencing. Default is 1.
    
    Returns:
    - result (numpy array): The differenced series.
    """
    
    result = np.array(series)
    for _ in range(order):
        result = np.diff(result)
    return result

In [ ]:
# Get the price data of the index
sp500_df = get_df("^GSPC", current_date, max_period=True)
sp500_df["Return"] = sp500_df["Close"].pct_change()
sp500_df["Volatility"] = sp500_df["Return"].rolling(window=252).std() * np.sqrt(252)

In [ ]:
# Test the stationarity of the closing price
sp500_closes = sp500_df["Close"].dropna()
is_close_stationary = adf_test(sp500_closes, "S&P 500 Closing Price")

# Test the stationarity of the return series
sp500_returns = sp500_df["Return"].dropna()
is_return_stationary = adf_test(sp500_returns, "S&P 500 Daily Return")

# Test the stationarity of the first difference of the closing price
sp500_closes_diff = difference_series(sp500_closes, order=1)
is_close_diff_stationary = adf_test(sp500_closes_diff, "S&P 500 Closing Price (1st Difference)")

In [ ]:
def plot_acf_pacf(acf_values, pacf_values, acf_confint, pacf_confint, title, nlags=40):
    """"
    Function to plot ACF and PACF with confidence intervals.

    Parameters:
    - acf_values (numpy array): ACF values.
    - pacf_values (numpy array): PACF values.
    - acf_confint (numpy array): ACF confidence intervals.
    - pacf_confint (numpy array): PACF confidence intervals.
    - title (str): Title for the plots.
    - nlags (int, optional): Number of lags to plot. Default is 40.

    Returns:
    - None
    """

    # Create lags array
    lags = np.arange(nlags + 1)

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Plot ACF
    ax1.stem(lags, acf_values, linefmt="b-", markerfmt="bo", basefmt="r-")
    ax1.axhline(y=0, color="black", linestyle="--")
    ax1.axhline(y=acf_confint[1, 0], color="red", linestyle="--")
    ax1.axhline(y=acf_confint[1, 1], color="red", linestyle="--")
    ax1.fill_between(lags, acf_confint[1, 0], acf_confint[1, 1], alpha=0.2, color="red")
    ax1.set_title(f"Autocorrelation Function (ACF) of {title}", fontsize=14)
    ax1.set_xlabel("Lag", fontsize=12)
    ax1.set_ylabel("ACF", fontsize=12)

    # Plot PACF
    ax2.stem(lags, pacf_values, linefmt="b-", markerfmt="bo", basefmt="r-")
    ax2.axhline(y=0, color="black", linestyle="--")
    ax2.axhline(y=pacf_confint[1, 0], color="red", linestyle="--")
    ax2.axhline(y=pacf_confint[1, 1], color="red", linestyle="--")
    ax2.fill_between(lags, pacf_confint[1, 0], pacf_confint[1, 1], alpha=0.2, color="red")
    ax2.set_title(f"Partial Autocorrelation Function (PACF) of {title}", fontsize=14)
    ax2.set_xlabel("Lag", fontsize=12)
    ax2.set_ylabel("PACF", fontsize=12)

    # Adjust layout
    plt.tight_layout()

    # Show the plots
    plt.show()

# Calculate ACF and PACF for the returns
nlags = 40
acf_values, acf_confint = acf(sp500_closes_diff, alpha=0.05, nlags=nlags, fft=True)
pacf_values, pacf_confint = pacf(sp500_closes_diff, alpha=0.05, nlags=nlags)

# Create DataFrame with PACF values and confidence intervals
pacf_df = pd.DataFrame({
    "Lag": np.arange(nlags + 1),
    "PACF": pacf_values,
    "Lower CI": pacf_confint[:, 0],
    "Upper CI": pacf_confint[:, 1]
})

# Plot ACF and PACF
plot_acf_pacf(acf_values, pacf_values, acf_confint, pacf_confint, "S&P 500 Daily Returns", nlags=nlags)

# Display the PACF DataFrame
display(pacf_df.head(10))

In [ ]:
def find_optimal_arima(data, p_values=np.arange(0, 5, 1), d_values=[0, 1], q_values=np.arange(0, 5, 1)):
    """
    Find the optimal ARIMA model based on AIC.

    Parameters:
    - data (pd.Series): Time series data.
    - p_values (list): List of AR terms to try. Default is [0, 1, 2].
    - d_values (list): List of differencing terms to try. Default is [0, 1].
    - q_values (list): List of MA terms to try. Default is [0, 1, 2].

    Returns:
    - best_model (ARIMAResults): Fitted ARIMA model with the lowest AIC.
    - best_order (tuple): Tuple of (p, d, q) for the best model.
    - best_aic (float): AIC of the best model.

    Notes:
    - Ensure data stationarity; high d values (e.g., > 2) may over-difference.
    - Warnings may occur for non-converging models; check output for details.
    """
    
    # Validate input
    if not isinstance(data, pd.Series):
        raise TypeError("Data must be a pandas Series.")

    # Initialize variables to store the best model
    best_aic = float("inf")
    best_order = None
    best_model = None
    
    for p, d, q in tqdm(itertools.product(p_values, d_values, q_values), desc="Fitting ARIMA models", total=len(p_values) * len(d_values) * len(q_values)):
        try:
            # Fit ARIMA model
            model = ARIMA(data, order=(p, d, q))
            results = model.fit()
            current_aic = results.aic

            # Check if current model is better
            if current_aic < best_aic:
                best_aic = current_aic
                best_order = (p, d, q)
                best_model = results
                
            print(f"ARIMA({p}, {d}, {q}) - AIC: {current_aic:.2f}")
            
        except Exception as e:
            print(f"ARIMA({p}, {d}, {q}) failed: {str(e)}")
            continue
    
    if best_model is None:
        raise ValueError("No ARIMA model successfully fitted. Check data or parameters.")
        
    return best_model, best_order, best_aic

# Prepare data
data = sp500_closes
forecast_length = 20
train_end = len(data) - forecast_length

# Split data into training and testing sets
train_data = data[:train_end]
test_data = data[train_end:]

# Define ARIMA parameters
p_values = np.arange(0, 5, 1)
d_values = [1]
q_values = np.arange(0, 5, 1)

# Find optimal ARIMA model
train_data = data[:train_end]
find_optimal = False
if find_optimal:
    best_model, best_order, best_aic = find_optimal_arima(
        train_data, 
        p_values=p_values, 
        d_values=d_values, 
        q_values=q_values
    )
else:
    best_order = (4, 1, 4)
    model = ARIMA(train_data, order=best_order)
    results = model.fit()
    best_model = results
    best_aic = results.aic

# Print ARIMA results
print("\nBest ARIMA Model Results (Initial Training):")
print(f"Optimal ARIMA order: {best_order}")
print(f"Best AIC: {best_aic:.2f}")
print("\nARIMA Model Summary:")
print(best_model.summary())

In [ ]:
def plot_residual(resid):
    """
    Function to plot residuals.

    Parameters:
    - resid (pd.Series): Residuals from ARIMA model.

    Returns:
    - None
    """

    # Create a figure with 4 subplots
    plt.figure(figsize=(12, 8))
    
    # Top-left subplot - Residuals over time
    plt.subplot(2, 2, 1)
    plt.plot(resid, color="blue")
    plt.axhline(y=0, color="red", linestyle="--")
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Residuals", fontsize=12)
    plt.title("Residuals of ARIMA Model", fontsize=14)
    
    # Top-right subplot - Histogram of residuals
    plt.subplot(2, 2, 2)
    plt.hist(resid, bins=30, color="blue", alpha=0.7)
    plt.axvline(x=0, color="red", linestyle="--")
    plt.xlim(resid.min(), resid.max())
    plt.xlabel("Residuals", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.title("Histogram of Residuals", fontsize=14)
    
    # Bottom-left subplot - ACF of residuals
    plt.subplot(2, 2, 3)
    sm.qqplot(resid, line="s", ax=plt.gca())
    plt.title("Q-Q Plot of Residuals", fontsize=14)

    # Adjust the layout
    plt.tight_layout()

    # Show the plot
    plt.show()

    # Plot the ACF and PACF of residuals
    acf_resid = acf(resid, alpha=0.05, nlags=40, fft=True)
    pacf_resid = pacf(resid, alpha=0.05, nlags=40)
    plot_acf_pacf(acf_resid[0], pacf_resid[0], acf_resid[1], pacf_resid[1], "ARIMA Residuals", nlags=40)

# Calculate ARIMA residuals
arima_resid = best_model.resid

# Plot the residuals
plot_residual(arima_resid)

# Perform Ljung-Box test on residuals
ljung_box_arima = acorr_ljungbox(arima_resid, lags=[10], return_df=True)
print("Ljung-Box Test Results:")
print(ljung_box_arima)

# Perform ARCH test on residuals
arima_arch_result = het_arch(arima_resid, ddof = 4)[1]
print("LM-test-Pvalue:", "{:.5f}".format(arima_arch_result))

In [ ]:
# Define GARCH model order
garch_order = (1, 1)
p, q = garch_order

# Fit GARCH model to residuals
garch_model = arch_model(arima_resid, vol="GARCH", p=p, q=q, dist="normal")
garch_result = garch_model.fit(disp="off")

# Print summary
print("\nGARCH Model Summary:")
print(garch_result.summary())

# Calculate residuals from GARCH model
garch_resid = garch_result.resid

# Calculate standardized residuals
garch_std_resid = garch_result.resid / garch_result.conditional_volatility

# Plot standardized residuals
plt.figure(figsize=(12, 6))
plt.plot(garch_std_resid, color="blue")
plt.axhline(y=0, color="red", linestyle="--")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Standardized Residual", fontsize=12)
plt.title("Standardized Residuals from ARIMA-GARCH Model", fontsize=14)
plt.tight_layout()
plt.show()

# Test for remaining ARCH effects
arch_arima_garch_result = het_arch(garch_std_resid, ddof=4)[1]
print("LM-test-Pvalue:", "{:.5f}".format(arch_arima_garch_result))

# Perform Ljung-Box test on GARCH residuals
ljung_box_arima_garch = acorr_ljungbox(garch_std_resid, lags=[10], return_df=True)
print("Ljung-Box Test Results for GARCH Residuals:")
print(ljung_box_arima_garch)

In [ ]:
def arima_garch_forecast(data, train_end, arima_order, garch_order):
    """"
    Function to perform rolling ARIMA-GARCH forecast.

    Parameters:
    - data (pd.Series): Time series data.
    - train_end (int): Index to split training and testing data.
    - arima_order (tuple): ARIMA order (p, d, q).
    - garch_order (tuple): GARCH order (p, q).

    Returns:
    - forecast_series (pd.Series): Forecasted values.
    - volatility_series (pd.Series): Forecasted volatilities.
    - lower_series (pd.Series): Lower bounds of prediction intervals.
    - upper_series (pd.Series): Upper bounds of prediction intervals.
    - resids (pd.Series): Residuals from the model.
    - full_fitted (pd.Series): Combined historical fitted values and forecasted values.
    """

    # Rolling forecast
    forecasts = []
    volatilities = []
    lower_bounds = []
    upper_bounds = []

    for i in tqdm(range(train_end, len(data)), desc="Rolling Forecast", total=len(data) - train_end):
        # ARIMA forecast
        train_window = data[:i]
        model = ARIMA(train_window, order=arima_order)
        results = model.fit()
        forecast = results.forecast(steps=1).iloc[0]
        forecasts.append(forecast)
        
        # Extract ARIMA residuals
        arima_resid = results.resid

        # Scale ARIMA residuals to avoid convergence issues
        arima_resid_scaled = arima_resid * 100
        
        # Fit GARCH model to ARIMA residuals
        p, q = garch_order
        garch_model = arch_model(arima_resid_scaled, vol="GARCH", p=p, q=q, mean="Zero", dist="normal")
        garch_result = garch_model.fit(disp="off")
        garch_forecast = garch_result.forecast(horizon=1)
        volatility = np.sqrt(garch_forecast.variance.iloc[-1].values[0]) / 100
        volatilities.append(volatility)
        
        # Calculate 95% prediction intervals
        lower_bound = forecast - 1.96 * volatility
        upper_bound = forecast + 1.96 * volatility
        lower_bounds.append(lower_bound)
        upper_bounds.append(upper_bound)

    # Convert to Series
    forecast_index = data.index[train_end:]
    forecast_series = pd.Series(forecasts, index=forecast_index)
    volatility_series = pd.Series(volatilities, index=forecast_index)
    lower_series = pd.Series(lower_bounds, index=forecast_index)
    upper_series = pd.Series(upper_bounds, index=forecast_index)
    
    return forecast_series, volatility_series, lower_series, upper_series

# Perform rolling forecast
forecast_series, volatility_series, lower_series, upper_series = arima_garch_forecast(data, train_end, best_order, garch_order)

# Historical fit values from initial ARIMA
hist_fit = best_model.fittedvalues

# Combine historical fit values and forecasted values
full_fit = pd.concat([hist_fit, forecast_series])

# Calculate residuals
resids = data - full_fit

# Performance metrics
mse_forecast = np.mean((data[train_end:] - forecast_series) ** 2)
mae_forecast = np.mean(np.abs(data[train_end:] - forecast_series))
print(f"Mean Squared Error (MSE): {mse_forecast:.2f}")
print(f"Mean Absolute Error (MAE): {mae_forecast:.2f}")
print(f"Average Forecast Volatility: {volatility_series.mean():.4f}")

Rolling Forecast:   5%|▌         | 1/20 [00:05<01:43,  5.45s/it]

In [ ]:
def plot_arima_garch_forecast(data, label, forecast_series, lower_series, upper_series, full_fit, arima_order, garch_order, window=60):
    """
    Function to plot ARIMA-GARCH forecast.

    Parameters:
    - data (pd.Series): Original time series data.
    - label (str): Label for the plot.
    - forecast_series (pd.Series): Forecasted values.
    - lower_series (pd.Series): Lower bounds of prediction intervals.
    - upper_series (pd.Series): Upper bounds of prediction intervals.
    - full_fit (pd.Series): Combined historical fitted values and forecasted values.
    - arima_order (tuple): ARIMA order (p, d, q).
    - garch_order (tuple): GARCH order (p, q).
    - window (int, optional): Window size for rolling forecast. Default is 60.

    Returns:
    - None
    """

    # Filter data for plotting
    data = data[- window:]
    full_fit = full_fit[- window:]

    # Plot the actual prices and forecasts on the top subplot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Top subplot - forecast
    ax1.plot(data, label="Actual Prices", color="blue")
    ax1.plot(full_fit, label="Fitted/Forecast (ARIMA)", color="green", linestyle="--")
    ax1.plot(forecast_series, color="green", linestyle="--")
    ax1.fill_between(forecast_series.index, lower_series, upper_series, color="green", alpha=0.2, label="95% Prediction Interval (GARCH)")
    ax1.axvline(x=forecast_series.index[0], color="red", linestyle=":", label="Forecast Start")
    ax1.legend()
    ax1.set_xlabel("Date", fontsize=12)
    ax1.set_ylabel(f"{label}", fontsize=12)
    ax1.set_title(f"ARIMA{arima_order} + GARCH{garch_order} - {label} Forecast", fontsize=14)

    # Calculate residuals
    resids = data - full_fit

    # Bottom subplot - residuals
    ax2.plot(resids, label="Residuals", color="purple")
    ax2.axhline(y=0, color="black", linestyle="-", alpha=0.3)
    ax2.legend()
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Residual Value")
    ax2.set_title("Forecast Residuals")
    
    # Set common x-axis limits
    buffer = pd.DateOffset(days=1)
    ax1.set_xlim(data.index.min() - buffer, data.index.max() + buffer)
    
    # Adjust the spacing
    plt.tight_layout()
    
    # Show the plot
    plt.show()

# Plot the forecast
plot_arima_garch_forecast(data, "S&P 500 Closing Price", forecast_series, lower_series, upper_series, full_fit, best_order, garch_order, window=60)